# Bölüm 20 · Ölçme Değişmezliği

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Değişmezlik düzeyini izin verdiği karşılaştırmayla eşler.
2. Gizil farkın referans birimini açıklar.
3. Kısmi değişmezlik ve çapa duyarlılığını raporlar.

**İleri rota:** Ortalama yapısı, çapa seçimi ve seçilim sonrası belirsizlik; Bölüm 19 sonrasında.

**Ç1 · Çıkış sorusu:** Yalnız metrik değişmezlik desteklenince gizil ortalama farkı otomatik olarak karşılaştırılabilir mi?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b20-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b20-olcme-degismezligi'
modules=['uygulamalar/b20-olcme-degismezligi/capa-duyarliligi']
expected_main=json.loads((root/"bolumler/b20/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 20 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## 3 · Ortak faktör analizi girdilerini üret

Bu bölüm Bölüm 18'in sabit A/B bölmesini ve ortak veri dosyalarını kullanır. Aşağıdaki adım aynı kaynaktan bunları yeniden üretir; başka bir defter açmanız gerekmez. Bu işlem yeni örneklem veya dış doğrulama oluşturmaz.

In [ ]:
run_script("bolumler/b18/analiz.py")

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b20/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 20 — görevler

Güncel Revizyon 23 görevleri; önce kendiniz çözün.

## D1–D3: Analizden Önce

1.  Neden standartlaştırılmış değil ham madde puanları kullanılır?

2.  Grup büyüklüklerinin (425 ve 891) dengesiz olması hangi yorumları etkileyebilir?

3.  Değişmezlik düzeylerini ve kısmi değişmezlik kuralını neden sonuçları görmeden yazmak gerekir?

Yanıtlar Ek F, Bölüm 20’dedir (s. kitaptaki ilgili sayfa).

# Rehberli ve Bağımsız Etkinlikler

## G1–G4: Rehberli Çözüm

1.  Biçimsel, metrik, skaler ve katı modeller için serbest parametre sayısını ve sd’yi hesaplayın.

2.  Metrik–skaler karşılaştırmasında $\Delta\chi^2$, $p$ ve $\Delta$CFI’yi yorumlayın. İki ölçüt neden farklı yönlere işaret ediyor?

3.  Metrik modelde A1’in kesişim farkı .505, yükü .387’dir; A3’ün farkı .461, yükü 1.045’tir. İkisinin ima ettiği gizil farkı hesaplayıp karşılaştırın.

4.  Tam ve kısmi skaler modellerin A farkını karşılaştırın. Bu karşılaştırma neyi sınar?

## B1–B3: Bağımsız Çözüm

1.  Kısmi değişmezlik kuralını $p<.01$ olarak yazsaydık hangi madde de serbest bırakılırdı? Bu durumda A faktöründe kaç değişmez madde kalır?

2.  Artık-varyans eşitliğinin reddedilmesi gözlenen toplam puanlarla yapılan bir t-testinin yorumunu nasıl etkiler? Ortalamanın yorumu ile standart hatanın hesabını ayırın.

3.  C faktöründe gözlenen $d$ (.18) ile gizil fark (.21) neden farklıdır?

## H1: Hatalı Raporu Onarma

Beş iddiayı düzeltin.

## P1: Uyarlama Çalışmasında Değişmezlik Planı

Bir ölçeğin Türkçe uyarlaması ile özgün form arasında değişmezlik sınayacak bir plan yazın: gruplar ve örneklemler, ölçüm modeli, veri türü (sürekli/ordinal) ve kestirici, ölçekleme, düzeyler ve karşılaştırma ölçütleri, kısmi değişmezlik kuralı, DIF bulgusu için içerik incelemesi ve raporlanacak tablolar. Rubrik: tasarım 20, model ve kestirim 25, ölçütler ve kurallar 25, yorum 20, yeniden üretim 10 puan.

## T1–T3: Çıpa, Birim ve Belirsizlik

1.  A1 ve A1+A2 serbest senaryolarının A farklarını ve aralıklarını karşılaştırın. Arama kuralı önceden yazılsa bile neden seçim belirsizliği kalır?

2.  Referans kadın grubu olduğunda A farkı neden yalnız $-.488$ olmaz? Yük işaretleyicisi ile kesişim çıpasının görevini ayırın.

3.  A ortalama puanının model farkını kesişim ve gizil bileşenlerine ayırın. Gözlenen farkla kalan artığı ve bunların neden nedensel etkiler olmadığını açıklayın.

P1 planına çıpa duyarlılığını, seçim sonrası çıkarım sınırını, referans birimini ve ham/model fark ayrıştırmasını ekleyin.

- Değişmezlik sınamadan gözlenen ölçek puanlarıyla grup karşılaştırması yapmak.

- Yalnız ki-kare farkına veya yalnız $\Delta$CFI’ye dayanmak; ölçütü sonuçlara göre seçmek.

- Kısmi değişmezlikte kesişimleri kural olmadan, uyum iyileşene kadar serbest bırakmak.

- Değişmez olmayan bir maddeyi işaretleyici madde olarak kullanmak.

- DIF bulgusunu içerik incelemesi yapmadan “madde yanlıdır” diye yorumlamak.

# Alıştırmalar

## Kolay Düzey

1.  Ölçme değişmezliği neden grup karşılaştırmasının ön koşuludur?

2.  Metrik değişmezlik hangi parametreleri eşitler?

3.  Gizil ortalamaları karşılaştırmak için en az hangi değişmezlik düzeyi gerekir?

4.  Kısmi değişmezlik nedir?

5.  $\Delta$CFI ölçütü neden ki-kare fark testine ek olarak kullanılır?

## Orta Düzey

1.  Biçimsel model iyi uyum veriyor, metrik modelde $\Delta\mathrm{CFI}=-.025$. Ne sonuç çıkarırsınız?

2.  Bir maddenin kesişimi bir grupta yüksektir, yükü eşittir. Bu maddenin toplam puana etkisi nedir?

3.  Grup büyüklükleri 150 ve 1500’dür. Değişmezlik testlerinde hangi sorunlar beklenir?

4.  İşaretleyici madde değişmez değilse ne olur?

5.  Katı değişmezlik hangi ek eşitliği sınar? Gözlenen ortalamaları karşılaştırmak veya güvenirliklerin eşit olduğunu söylemek için koşulları açıklayın.

## İleri Düzey

1.  Farklı madde işlevi (DIF) ile ölçme değişmezliği arasındaki ilişkiyi açıklayın.

2.  Ordinal maddelerde eşik değişmezliği neden sürekli yaklaşımdaki kesişim değişmezliğinden farklıdır?

3.  Yirmi ülkeli bir karşılaştırmada ardışık çok gruplu DFA neden pratik değildir? Ne önerirsiniz?

4.  Kısmi değişmezlikte sorunlu maddeyi serbest bırakmak ile ölçekten çıkarmak arasındaki farkı tartışın.

5.  Güçlü bir değişmezlik raporunda bulunması gereken en az beş unsuru yazın.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b20/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Tam skaler veya A1 kesişimi serbest senaryoyu seçin; A farkını aynı referans biriminde karşılaştırın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 20,
    "gorev": 'Tam skaler veya A1 kesişimi serbest senaryoyu seçin; A farkını aynı referans biriminde karşılaştırın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":20,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b20-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b20-olcme-degismezligi',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)